# Sensitivity Analysis — PEM and AEC Electrolyzer Capacity

Given a fixed DRI trajectory (70% by 2050), which electrolyzer technology parameters drive PEM and AEC capacity?

- **PEM capacity** → directly determines iridium demand
- **AEC capacity** → directly determines nickel demand

DRI share is a scenario assumption and appears in the scenario comparison, not here.

In [ ]:
import sys
sys.path.append('..')
from model_functions import run_scenario
import matplotlib.pyplot as plt
import numpy as np
from SALib.sample import morris as morris_sample
from SALib.analyze import morris as morris_analyze
from SALib.plotting.morris import horizontal_bar_plot, covariance_plot

In [ ]:
# ── Iridium sensitivity ───────────────────────────────────────────────────────
problem_ir = {
    'num_vars': 6,
    'names': [
        'dri_share_2050',
        'pem_share',
        'lifetime_PEM',
        'h2_conversion',
        'iridium_intensity_2050',
        'L',
    ],
    'bounds': [
        [0.10, 1.00],    # dri_share_2050
        [0.20, 0.60],    # pem_share
        [65500, 105000], # lifetime_PEM (hours)
        [54,    67],     # h2_conversion (kg H2/t steel)
        [0.03,  0.30],   # iridium_intensity_2050 (g/kW)
        [6,     16],     # L (t/cap)
    ]
}

np.random.seed(42)
param_values_ir = morris_sample.sample(problem_ir, N=50, optimal_trajectories=None)

IDX_2050 = np.where(np.arange(2022, 2101) == 2050)[0][0]

baseline_ir = {
    'scrap_end':              0.85,
    'grid_co2':               'current_policies',
    'eaf_efficiency':         470,
    'aec_share':              0.55,
    'other_share':            0.10,
    'lifetime_AEC':           90000,
    'lifetime_Other':         45000,
    'efficiency_AEC':         50.0,
    'efficiency_PEM':         55.0,
    'efficiency_Other':       43.4,
    'nickel_intensity':       800,
    'platinum_intensity':     0.5,
    'iridium_intensity_2022': 0.75,
    'iridium_intensity_2030': 0.20,  # fixed at baseline
}

In [ ]:
Y_ir = np.zeros(len(param_values_ir))

for i, sample in enumerate(param_values_ir):
    p = baseline_ir.copy()
    p['dri_share_2050']         = sample[0]
    p['pem_share']              = sample[1]
    p['other_share']            = 1 - 0.55 - sample[1]
    p['lifetime_PEM']           = sample[2]
    p['h2_conversion']          = sample[3]
    p['iridium_intensity_2050'] = sample[4]
    p['L']                      = sample[5]

    sc = run_scenario(p, f'morris_ir_{i}', verbose=False)
    Y_ir[i] = sc['materials']['iridium_inflow'][IDX_2050] * 1000

print('Done.')

## Iridium Sensitivity

In [ ]:
Si_ir = morris_analyze.analyze(problem_ir, param_values_ir, Y_ir, print_to_console=False)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
horizontal_bar_plot(ax, Si_ir, unit='t/yr')
ax.set_title('Morris Sensitivity — Iridium inflow at 2050 (t/yr)')
plt.tight_layout()
plt.show()

# Covariance plot
fig, ax = plt.subplots(figsize=(6, 6))
covariance_plot(ax, Si_ir, unit='t/yr')
ax.set_title('Morris Sensitivity — Iridium inflow at 2050 (t/yr)')
plt.tight_layout()
plt.show()

## Nickel Sensitivity

In [ ]:
# ── Nickel sensitivity ────────────────────────────────────────────────────────
problem_ni = {
    'num_vars': 6,
    'names': [
        'dri_share_2050',
        'h2_conversion',
        'L',
        'nickel_intensity',
        'lifetime_AEC',
        'aec_share',
    ],
    'bounds': [
        [0.10, 1.00],    # dri_share_2050
        [54,   67],      # h2_conversion (kg H2/t steel)
        [6,    16],      # L (t/cap)
        [400, 1200],     # nickel_intensity (kg/MW)
        [65500, 105000], # lifetime_AEC (hours)
        [0.20, 0.70],    # aec_share (pem fixed at 0.20, other = 1 - aec - 0.20)
    ]
}

np.random.seed(42)
param_values_ni = morris_sample.sample(problem_ni, N=50, optimal_trajectories=None)

baseline_ni = {
    'scrap_end':              0.85,
    'grid_co2':               'current_policies',
    'eaf_efficiency':         470,
    'aec_share':              0.55,
    'pem_share':              0.20,
    'other_share':            0.25,
    'lifetime_AEC':           90000,
    'lifetime_PEM':           85000,
    'lifetime_Other':         45000,
    'efficiency_AEC':         50.0,
    'efficiency_PEM':         55.0,
    'efficiency_Other':       43.4,
    'nickel_intensity':       800,
    'platinum_intensity':     0.5,
    'iridium_intensity_2022': 0.75,
    'iridium_intensity_2030': 0.20,
    'iridium_intensity_2050': 0.10,
    'dri_share_2050':         0.70,
    'h2_conversion':          60,
    'L':                      11.3,
}

Y_ni = np.zeros(len(param_values_ni))

for i, sample in enumerate(param_values_ni):
    p = baseline_ni.copy()
    p['dri_share_2050']   = sample[0]
    p['h2_conversion']    = sample[1]
    p['L']                = sample[2]
    p['nickel_intensity'] = sample[3]
    p['lifetime_AEC']     = sample[4]
    p['aec_share']        = sample[5]
    p['other_share']      = 1 - sample[5] - 0.20

    sc = run_scenario(p, f'morris_ni_{i}', verbose=False)
    Y_ni[i] = sc['materials']['nickel_inflow'][IDX_2050]

print('Done.')

In [ ]:
Si_ni = morris_analyze.analyze(problem_ni, param_values_ni, Y_ni, print_to_console=False)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
horizontal_bar_plot(ax, Si_ni, unit='kt/yr')
ax.set_title('Morris Sensitivity — Nickel inflow at 2050 (kt/yr)')
plt.tight_layout()
plt.show()

# Covariance plot (μ* vs σ)
fig, ax = plt.subplots(figsize=(6, 6))
covariance_plot(ax, Si_ni, unit='kt/yr')
ax.set_title('Morris Sensitivity — Nickel inflow at 2050 (kt/yr)')
plt.tight_layout()
plt.show()

## CO₂ Emissions Sensitivity

In [ ]:
problem_co2 = {
    'num_vars': 6,
    'names': [
        'L',
        'dri_share_2050',
        'bfbof_share_2050',
        'grid_co2',
        'eaf_efficiency',
        'bf_emission_factor',
    ],
    'bounds': [
        [6,    16],    # L — steel intensity saturation (t/cap)
        [0.00, 0.60],  # dri_share_2050 — H-DRI share of total steel
        [0.05, 0.70],  # bfbof_share_2050 — BF-BOF share of total steel
        [0,    400],   # grid_co2 — grid CO2 intensity at 2050 (g/kWh)
        [400,  550],   # eaf_efficiency — EAF electricity consumption (kWh/t)
        [1.5,  2.3],   # bf_emission_factor — BF-BOF emission intensity (t CO2/t steel)
    ]
}

np.random.seed(42)
param_values_co2 = morris_sample.sample(problem_co2, N=50, optimal_trajectories=None)

baseline_co2 = {
    'scrap_end':              0.85,
    'grid_co2':               200,
    'eaf_efficiency':         470,
    'dri_share_2050':         0.30,
    'bfbof_share_2050':       0.40,
    'bf_emission_factor':     1.83,
    'aec_share':              0.55,
    'pem_share':              0.40,
    'other_share':            0.05,
    'lifetime_AEC':           90000,
    'lifetime_PEM':           85000,
    'lifetime_Other':         45000,
    'efficiency_AEC':         50.0,
    'efficiency_PEM':         55.0,
    'efficiency_Other':       43.4,
    'h2_conversion':          60,
    'nickel_intensity':       800,
    'platinum_intensity':     0.5,
    'iridium_intensity_2022': 0.75,
    'iridium_intensity_2030': 0.20,
    'iridium_intensity_2050': 0.10,
    'L':                      11.3,
}

Y_co2 = np.zeros(len(param_values_co2))

for i, sample in enumerate(param_values_co2):
    p = baseline_co2.copy()
    p['L']                  = sample[0]
    p['dri_share_2050']     = sample[1]
    p['bfbof_share_2050']   = sample[2]
    p['grid_co2']           = sample[3]
    p['eaf_efficiency']     = sample[4]
    p['bf_emission_factor'] = sample[5]

    sc = run_scenario(p, f'morris_co2_{i}', verbose=False)
    Y_co2[i] = sc['emissions']['Total Emissions'][IDX_2050]

print('Done.')

In [ ]:
Si_co2 = morris_analyze.analyze(problem_co2, param_values_co2, Y_co2, print_to_console=False)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
horizontal_bar_plot(ax, Si_co2, unit='Mt/yr')
ax.set_title('Morris Sensitivity — Annual CO₂ emissions at 2050 (Mt/yr)')
plt.tight_layout()
plt.show()

# Covariance plot (μ* vs σ)
fig, ax = plt.subplots(figsize=(6, 6))
covariance_plot(ax, Si_co2, unit='Mt/yr')
ax.set_title('Morris Sensitivity — Annual CO₂ emissions at 2050 (Mt/yr)')
plt.tight_layout()
plt.show()